# Direct Parquet serving for external code tools

Status: approved architecture; written specification awaiting review  
Date: 2026-08-27  
Design epic: `bd-1iru`  
Plan ID: `cf89a7ee-d13d-40ad-9a4f-b91ad8130847`  
Optimization evidence: `sol_e486929ddb034f79`

## Decision

Split external serving into exactly two directly addressed serving Lambdas:

- **Code Lambda:** `external_catalog`, `external_code_*`, `external_index`, and `external_index_status`; backed by a generation registry, Silver Parquet, and an immutable source-text sidecar.
- **Knowledge Lambda:** `external_knowledge_context_pack`; backed by the generation registry and the frozen DuckDB/DuckLake snapshot.

The caller dispatches by tool class, so every operation invokes exactly one serving Lambda. No routing Lambda is introduced.

## Context, goals, and scope

Today the context-service prepares a frozen DuckDB/DuckLake catalog before almost every external read tool. That is unnecessary for symbol lookup and graph traversal now that `spur_graph::ParquetClient` provides projected Parquet reads plus lazy hot symbol and adjacency indexes.

The remaining backend facts are:

- Silver graph artifacts contain files, symbols, and edges, but not file source text.
- `external_code_read` currently obtains source text from DuckDB `gold.files`.
- `external_knowledge_context_pack` genuinely requires DuckDB/DuckLake for BM25, vector candidates, documents, and ranking.
- Global package/revision/ref exploration needs a small routing catalog, not a SQL engine.

### Goals

- Preserve the public result shapes and semantics of every `external_*` tool.
- Remove DuckDB, DuckLake extensions, and catalog-secret handling from the Code Lambda artifact and runtime.
- Use the optimized native Parquet query path for symbol search, resolution, callers, and callees.
- Serve source reads from an immutable source-text sidecar.
- Keep one Lambda invocation per operation and exactly two **serving** Lambdas.
- Publish catalog, Silver graph, source text, and DuckLake metadata as one generation-consistent serving view.
- Fail closed on incomplete, corrupt, or generation-mismatched artifacts.

### Non-goals

- Rewriting the knowledge ranking algorithm or replacing DuckLake.
- Redesigning the source-fetch, Step Functions, ECS fallback, authorizer, cleanup, or indexing-worker pipeline.
- Claiming that the entire context-service deployment has only two Lambda resources; the two-Lambda constraint applies to serving ingress.
- Enabling provisioned concurrency by default.

## Approach selection and optimization evidence

Three serving architectures were evaluated with Z3 Optimize:

| Architecture | Code artifact has DuckDB | Invocations/request | Serving Lambdas | Caller route change |
|---|---:|---:|---:|---:|
| Single hybrid Lambda | 1 | 1 | 1 | 0 |
| Two direct Lambdas | 0 | 1 | 2 | 1 |
| Router + two backend Lambdas | 0 | 2 | 3 | 0 |

The complete lexicographic optimization minimized, in order: DuckDB in the code artifact, invocations per request, deployed serving Lambdas, and caller changes. With `caller_can_route_by_tool = true`, the exact optimum was `split_two_direct = [0, 1, 2, 1]`. Persisted evidence: `sol_e486929ddb034f79`.

A sensitivity solve with tool-aware caller routing disabled selected the three-Lambda router architecture. Therefore direct caller dispatch is a required eligibility condition, not an optional optimization.

Rejected alternatives:

- **Single hybrid Lambda:** one deployment remains simpler, but every code cold start still carries DuckDB and its extensions.
- **Routing Lambda:** preserves one opaque endpoint but adds an invocation, latency, IAM surface, logs, and a third serving function.
- **DuckDB views over Silver Parquet:** retains DuckDB initialization and bypasses the native hot indexes.

In [ ]:
architecture-beta
    %% @ns-zone-allow caller serving
    %% @ns-zone-allow serving storage
    group caller(cloud)[Caller]
    group serving(cloud)[Serving]
    group storage(cloud)[Immutable S3 Artifacts]
    service dispatcher(server)[Tool-aware MCP Dispatcher] in caller
    service code_lambda(server)[Code Lambda] in serving
    service knowledge_lambda(server)[Knowledge Lambda] in serving
    service registry(disk)[Generation and Catalog Registry] in storage
    service silver(disk)[Silver Parquet] in storage
    service source(disk)[Source Text Sidecar] in storage
    service ducklake(database)[DuckLake Snapshot] in storage
    dispatcher:R --> L:code_lambda
    dispatcher:R --> L:knowledge_lambda
    code_lambda:R --> L:registry
    code_lambda:R --> L:silver
    code_lambda:R --> L:source
    knowledge_lambda:R --> L:registry
    knowledge_lambda:R --> L:ducklake

## Components and immutable artifact contract

### Tool-aware caller dispatch

The existing caller maps tool names to one of two API Gateway routes on the same service hostname. `external_catalog`, `external_code_*`, `external_index`, and `external_index_status` target the Code Lambda route. `external_knowledge_context_pack` targets the Knowledge Lambda route. Authentication and throttling remain shared at API Gateway; no request is forwarded through another Lambda.

### Generation and catalog registry

A small immutable registry is published for every generation. A stable `current` pointer references exactly one complete registry. Required registry fields are:

- `schema_version`, `generation_id`, publication timestamp, and lineage/snapshot identity.
- DuckLake snapshot URI and SHA-256.
- For each source/package/revision: revision kind, refs/aliases, snapshot ID, Silver manifest URI and hash, and source-sidecar manifest URI and hash.

The registry is the global substrate for `external_catalog` source/package/revision/ref exploration. After selecting a package revision, file and symbol descent uses `ParquetClient`.

### Silver graph artifacts

Existing Silver Parquet remains authoritative for files, symbols, resolved edges, and unresolved edges. The Code Lambda adapts `GraphQueryClient` results to the existing MCP response structs; it does not duplicate query semantics.

### Source-text sidecar

Each package revision publishes immutable source data with at least `file_path`, `content_oid`, and `source_text`. `external_code_read` first resolves symbol metadata through `ParquetClient`, then performs an exact file-path lookup with column projection against the sidecar and slices the validated symbol range.

### Knowledge snapshot

The Knowledge Lambda uses the registry-selected frozen DuckDB/DuckLake snapshot. Existing BM25, vector, document, and confidence logic remains authoritative.

In [ ]:
flowchart TD
    SPEC["`@spec EXTERNAL-TOOL-ROUTING-V1
@type ToolClass = enum[catalog, code, index, knowledge]
@type Status = enum[code_lambda, knowledge_lambda]
@input tool: ToolClass
@output status: Status
@requires ELIGIBLE: true`"]
    CODE["`@branch CODE_SERVICE
@when tool = catalog or tool = code or tool = index
@ensures CODE_TARGET: status = code_lambda`"]
    KNOWLEDGE["`@branch KNOWLEDGE_SERVICE
@when tool = knowledge
@ensures KNOWLEDGE_TARGET: status = knowledge_lambda`"]
    CHECK["`@verify ROUTING_CONSISTENT: witness consistency
@verify ROUTING_DETERMINISTIC: prove determinism
@verify ROUTING_COVERED: prove partition_coverage
@verify ROUTING_EXCLUSIVE: prove partition_exclusive
@verify BOTH_TARGETS_REACHABLE: witness each status`"]
    SPEC --> CODE --> CHECK
    SPEC --> KNOWLEDGE --> CHECK

## Request flow and caching

### Code request

1. Load or refresh the stable generation pointer and immutable registry using ETag-aware caching.
2. Resolve source/package/revision/ref to immutable artifact identities.
3. Validate manifest completeness and declared hashes.
4. Materialize only the selected package artifacts under `/tmp`, keyed by content hash.
5. Reuse a warm `Arc<ParquetClient>` keyed by generation, package revision, and Silver content hash.
6. Execute search, resolve, caller/callee traversal, or exact source lookup.
7. Return the existing public response structure.

### Knowledge request

1. Resolve the current generation through the same registry contract.
2. Reuse or prepare the generation-specific frozen DuckDB snapshot.
3. Execute the existing knowledge-context query and ranking pipeline.

### Cache invariants

- Process caches and `/tmp` files are keyed by immutable content identity, never mutable package names alone.
- Downloaders coalesce concurrent misses and publish local files only after hash verification and atomic rename.
- Cache sizes are bounded by configured bytes and entry counts; eviction never deletes an artifact still borrowed by an active request.
- A generation switch creates new cache keys. Old entries may be evicted after in-flight requests release them.
- No process may combine registry, Silver, source, or DuckLake artifacts from different generation identities.

In [ ]:
flowchart TD
    SPEC["`@spec PUBLICATION-ELIGIBILITY-V1
@type Status = enum[serve, reject]
@input registry_complete: Bool
@input silver_complete: Bool
@input source_complete: Bool
@input ducklake_complete: Bool
@input generation_match: Bool
@input hash_match: Bool
@output status: Status
@requires ELIGIBLE_INPUT: true`"]
    SERVE["`@branch SERVE_GENERATION
@when registry_complete = true and silver_complete = true and source_complete = true and ducklake_complete = true and generation_match = true and hash_match = true
@ensures SERVE_STATUS: status = serve`"]
    REJECT["`@branch REJECT_GENERATION
@when registry_complete = false or silver_complete = false or source_complete = false or ducklake_complete = false or generation_match = false or hash_match = false
@ensures REJECT_STATUS: status = reject`"]
    CHECK["`@verify PUBLICATION_CONSISTENT: witness consistency
@verify PUBLICATION_DETERMINISTIC: prove determinism
@verify PUBLICATION_COVERED: prove partition_coverage
@verify PUBLICATION_EXCLUSIVE: prove partition_exclusive
@verify BOTH_RESULTS_REACHABLE: witness each status`"]
    SPEC --> SERVE --> CHECK
    SPEC --> REJECT --> CHECK

## Publication, errors, and security

### Publication protocol

1. The indexing publisher holds the existing publication lease.
2. Upload Silver, source-sidecar, and DuckLake payloads under an immutable generation prefix.
3. Verify manifests, completeness markers, and SHA-256 values.
4. Write the immutable generation registry.
5. Conditionally replace the stable `current` pointer last, guarded by the expected pointer ETag and active lease.

Readers serve the previously complete generation until step 5 succeeds. A failed publication never exposes a partially written generation.

### Error behavior

- Missing current pointer, incomplete manifest, hash failure, or generation mismatch returns a sanitized retryable catalog-unavailable error.
- Unknown package, revision, file, or symbol remains a normal not-found result.
- Artifact unavailability must not be converted into misleading empty search/catalog responses.
- Deadline, disk-budget, and memory-budget failures remain bounded and sanitized.

### Security boundaries

- Code Lambda IAM permits read access only to the generation registry, Silver/source prefixes, and the existing index-control resources it invokes.
- Knowledge Lambda IAM permits registry and DuckLake snapshot access plus only the catalog secret permissions required by its backend.
- Both functions retain least-privilege logs, X-Ray, API authentication, bounded JSON logging, and VPC controls where required.
- AWS SDK credential resolution remains refreshable. DuckDB temporary-secret construction exists only in the Knowledge Lambda.
- Logs and responses never expose credentials, SQL containing secrets, presigned URLs, or raw backend errors.

## Deployment, concurrency, and observability

Two independently built serving artifacts are deployed:

- **Code image:** context-service MCP/API shell, AWS clients, registry resolver, `spur-graph`, Parquet/Arrow readers, and source-sidecar adapter. It must not link DuckDB or bundle DuckLake extensions.
- **Knowledge image:** context-service MCP/API shell, AWS clients, DuckDB/DuckLake extensions, credential bridge, and knowledge query/ranking implementation.

API Gateway exposes distinct direct integrations while retaining the public tool names. The caller owns tool-to-route selection. Existing index orchestration and worker resources remain unchanged.

Reserved concurrency may cap each function independently and does not prewarm capacity. Provisioned concurrency is disabled by default because it creates ongoing charges. The split must preserve the account-level concurrency budget rather than silently doubling it.

Required dimensions and alarms include function/backend, tool name, generation ID, cold/warm invocation, registry refresh, artifact cache hit/miss, download bytes/time, Parquet query time, DuckDB preparation time, result status, deadline exhaustion, and integrity rejection. Cost reporting must distinguish Code and Knowledge Lambda duration.

## Migration and rollback

1. Extend publication to emit the source sidecar and immutable generation registry while the current serving path remains authoritative.
2. Implement the Code Lambda adapter and run response-parity tests against the current DuckDB implementation using identical frozen fixtures.
3. Extract the Knowledge Lambda entrypoint without changing knowledge query semantics.
4. Add direct API routes and tool-aware caller dispatch behind deployment configuration.
5. In staging, run AWS CLI and external-tool smoke tests for catalog, search, read, callers, callees, index/status, and knowledge context.
6. Cut over direct routing after integrity, parity, latency, and cost gates pass.

Rollback changes caller routing back to the current serving endpoint and retains the last complete generation pointer. Immutable artifacts are not deleted during rollback. The publication contract remains backward-compatible for at least the rollback window.

## Verification and acceptance criteria

### Required tests

- Unit tests for complete tool routing, revision/ref resolution, registry validation, hash checking, source slicing, cache identity, coalesced downloads, eviction safety, and sanitized errors.
- Contract tests proving every tool maps to exactly one backend and that no Code Lambda dependency enables DuckDB.
- Golden parity tests comparing old DuckDB and new Parquet results for catalog, search, read, callers, and callees over the same published fixture.
- Corruption tests for missing manifests, mismatched hashes, incomplete sidecars, stale pointers, and mixed generations.
- Integration tests covering S3 registry refresh, warm-client reuse, deadline interruption, and concurrent requests.
- Staging AWS CLI plus `external_*` smoke tests against both direct routes.
- Cold/warm performance and cost measurements before production cutover.

### Release gates

- All formal notebook obligations execute with their expected statuses and fresh source hashes.
- Every supported tool returns public-contract-compatible results.
- One and only one serving Lambda is invoked for each operation.
- The Code image contains no DuckDB library or DuckLake extension.
- Every served response is attributable to one complete generation.
- Any incomplete, corrupt, or mismatched generation fails closed.
- No provisioned concurrency is enabled without a separate measured cost decision.
- Rollback to the previous serving endpoint and generation pointer is rehearsed in staging.

## Implementation-plan boundaries

The implementation plan should preserve these dependency boundaries:

1. **Artifact contract and publisher:** generation registry, source sidecar, pointer protocol, and fixtures.
2. **Code backend:** registry resolver, artifact cache, `ParquetClient` adapter, source reads, and response parity; depends on the artifact contract.
3. **Knowledge backend:** isolated entrypoint/image using the registry-selected DuckLake snapshot; depends on the registry contract and may proceed in parallel with the Code backend.
4. **Ingress and infrastructure:** direct API integrations, caller routing, IAM, deployment packaging, concurrency, and observability; depends on both backend entrypoint contracts.
5. **Cross-backend verification:** corruption, AWS CLI, external-tool parity, performance, cost, cutover, and rollback; depends on all prior tasks.

No implementation work begins from this notebook until the written specification is reviewed and the design epic transitions to an approved state.

## Formal evidence index

- `ARCHITECTURE` — cell `516b4f94-8455-46d8-b949-f9dbc25a4d7f`; 8/8 obligations matched; report hash `24aa123e84c44685e5b1f8e663bb71f681f30ade51a2c00d194438a73b621abc`.
- `EXTERNAL-TOOL-ROUTING-V1` — cell `6764ac8d-d138-433b-a6be-62af4470d36d`; 6/6 obligations matched; report hash `e9b337946bd27e520613377df3be7a27b79e04980044c5fff1e1807bbd509b9f`.
- `PUBLICATION-ELIGIBILITY-V1` — cell `e28788f2-13d0-4e36-96e9-309692648b9a`; 6/6 obligations matched; report hash `fdb10173d9db00959567eb6808b319b4dda6474c0e9f42772e709d62cfc897a1`.
- Architecture optimization — persisted solve `sol_e486929ddb034f79`; complete lexicographic optimum `split_two_direct`.

All native proof reports are solver-verified, binding-complete, obligation-complete, and source-fresh.